# **Example usage of Clustered Echo State Network using Ring, Erdős–Rényi, and Scale-Free Network**


**Import necessary packages**

`cesn` contains four classes:

- `EchoStateNetwork` — Basic Echo State Network
- `EchoStateNetworkWithBAClusters` — Scale-Free Network
- `EchoStateNetworkWithErdosRenyiClusters` — Erdős–Rényi Network
- `EchoStateNetworkWithAdaptiveClusters` — Ring Network

`utils_func'`contains these following functions:

- `train_test_split`
- `compute_nrmse`
- `average_state_change`
- `plot_state_correlation_matrix`

In [ ]:
from utils_func import *  #specific functions can be imported
from cesn import *   #specific class function can be imported

import os
import matplotlib as mpl
from scipy.signal import savgol_filter
import matplotlib.ticker as ticker
import numpy as np
from scipy.linalg import pinv
import matplotlib.pyplot as plt
import networkx as nx  # For network generation and visualization
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import time
import tracemalloc
from scipy.ndimage import gaussian_filter1d

**Import time series**

In [ ]:
# Construct file path relative to script location
base_path = os.getcwd()  # current working directory
file_path = os.path.join(base_path, 'data', 'multi_series.txt')


# Define column names based on the file format
columns = [
    "Timestamp", "Time", "Source", "Bt-med", "Bt-min", "Bt-max",
    "Bx-med", "Bx-min", "Bx-max", "By-med", "By-min", "By-max",
    "Bz-med", "Bz-min", "Bz-max", "Phi-mean", "Phi-min", "Phi-max",
    "Theta-med", "Theta-min", "Theta-max", "Dens-med", "Dens-min",
    "Dens-max", "Speed-med", "Speed-min", "Speed-max", "Temp-med",
    "Temp-min", "Temp-max"
]

# Load the text file into a pandas DataFrame with the correct column names
df = pd.read_csv(file_path, delim_whitespace=True, names=columns, skiprows=14)

# Combine 'Timestamp' and 'Time' into a single datetime column
df['Datetime'] = pd.to_datetime(df['Timestamp'] + ' ' + df['Time'])

# Filter out rows where 'Dens-max', 'Speed-max', or 'Temp-max' are negative
df_filtered = df[(df['Dens-max'] > 0) & (df['Speed-max'] > 0) & (df['Temp-max'] > 0)]

**Data Preparation**

- **Scaling and filtering of data**
- **Split of Training and Test Data**

In [ ]:
# Extract relevant columns for plotting
times = df_filtered['Datetime']
dens_max = df_filtered['Dens-max']
speed_max = df_filtered['Speed-max']
temp_max = df_filtered['Temp-max']


x1 = df_filtered['Dens-max'].values
x2 = df_filtered['Speed-max'].values
x3 = df_filtered['Temp-max'].values

# Initialize the MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1.0))

# Combine x1 and x2 to scale them together
x_combined = np.column_stack((x1, x2,x3))

# Fit the scaler and transform the data
x_scaled = scaler.fit_transform(x_combined)

# Separate the scaled x1 and x2
x1_scaled = x_scaled[:, 0]
x2_scaled = x_scaled[:, 1]
x3_scaled = x_scaled[:, 2]




# Apply Gaussian filter with sigma = 2
x1_smoothed_gauss = gaussian_filter1d(x1_scaled, sigma=1)
x2_smoothed_gauss = gaussian_filter1d(x2_scaled, sigma=1)
x3_smoothed_gauss = gaussian_filter1d(x3_scaled, sigma=1)

X = np.column_stack((x1_smoothed_gauss[:-1], x2_smoothed_gauss[:-1],x3_smoothed_gauss[:-1]))  # Inputs: x1 and x2 values
y = np.column_stack((x1_smoothed_gauss[1:], x2_smoothed_gauss[1:],x3_smoothed_gauss[1:]))  # Outputs: shifted x1 and x2 values

X = X[:-1]
y = y[:-1]

# Adjust dates to match inputs (X) and outputs (y)
dates = df_filtered['Datetime'].iloc[:-2].reset_index(drop=True)

# Define your train ratio (e.g., 80% training, 20% testing)
train_ratio = 0.8

# Split the data using the generalized function
X_train, y_train, X_test, y_test, train_dates, test_dates = train_test_split(X, y, dates, train_ratio)

# Check the shapes of the resulting arrays
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")
print(f"train_dates length: {len(train_dates)}, test_dates length: {len(test_dates)}")

**Defining Parameters**

Intra-cluster connectivity parameter(p_in):

- **Ring:** 1.0
- **Erdos Renyi:** Ratio
- **Scale free:** Mean degree - [1,10]

In [ ]:
# Define parameters
n_clusters = X_train.shape[1]  # Number of clusters equals the number of input dimensions (oscillators)
neurons_per_cluster = 100  # Fixed number of neurons per cluster
spectral_radius = 0.9
input_scaling = 1.0
p_in = 1.0  # For ring (p_in = 1), ER -> Ratio, SF: Mean degree - [1,10]
p_out = 0.01  # Sparse inter-cluster connection
input_connectivity = 0.2
ridge_lambda = 1e-8  # Regularization parameter
leak_rate = 1.0
seed = 12
washout =100

**Construction of Reservoir Network**

-Three classes can be used as defined below

In [ ]:
# Start tracing memory
tracemalloc.start()


# esn = EchoStateNetworkWithAdaptiveClusters(
#     n_inputs=n_clusters,
#     neurons_per_cluster=neurons_per_cluster,
#     spectral_radius=spectral_radius,
#     leak_rate=leak_rate,
#     input_scaling=input_scaling,
#     intra_cluster_connectivity=p_in,
#     inter_cluster_connectivity=p_out,
#     input_connectivity=input_connectivity,
#     seed=seed)

# esn = EchoStateNetworkWithErdosRenyiClusters(
#     n_inputs=n_clusters, neurons_per_cluster=neurons_per_cluster, 
#     spectral_radius=spectral_radius, leak_rate=leak_rate, 
#     input_scaling=input_scaling, p_in=p_in, p_out=p_out, 
#     input_connectivity=input_connectivity, seed=seed)

start_train = time.time()


esn = EchoStateNetworkWithBAClusters(
    n_inputs=n_clusters, neurons_per_cluster=neurons_per_cluster, 
    n_clusters=n_clusters, spectral_radius=spectral_radius, 
    leak_rate=leak_rate, input_scaling=input_scaling, 
    mean_degree=p_in, inter_cluster_connectivity=p_out, 
    input_connectivity=input_connectivity, seed=seed)

# Train and evaluate
esn.fit(X_train, y_train, ridge_lambda=ridge_lambda, reset=True,washout=washout)


end_train = time.time()
train_time = end_train - start_train
print(f"Training Time: {train_time:.6f} seconds")


y_pred = esn.predict(X_test, reset=True,washout=washout)
# Get current and peak memory
current, peak = tracemalloc.get_traced_memory()
print(f"Current memory usage: {current / 1e6:.2f} MB")
print(f"Peak memory usage: {peak / 1e6:.2f} MB")

# Stop tracing
tracemalloc.stop()

**Computation of NRMSE**

-Since it is a multivariate time series, one can compute averaged NRMSE as below.

In [ ]:
nrmse = compute_nrmse(y_pred, y_test[washout:])
avg_nrmse = np.mean(nrmse)
print(avg_nrmse)

**Renormalization**

In [ ]:
# Assuming y_pred is the prediction output from your ESN in normalized scale:
y_pred_denorm = scaler.inverse_transform(y_pred)

# Similarly, if you want to de-normalize the actual test outputs:
y_test_denorm = scaler.inverse_transform(y_test[washout:])


**Plotting the Original and Predicition curves**

In [ ]:
# Variable names
variable_names = ['Density', 'Speed', 'Temperature']

# Number of features
n_features = X_train.shape[1]

# Create subplots
fig, axes = plt.subplots(n_features, 1, figsize=(12, 12), sharex=True)

for i in range(n_features):

    # Plot the test set for each feature
    axes[i].plot(test_dates[washout:], y_test_denorm[:, i], label='Test Set', color='orange', linewidth=2)

    # Plot the predicted set with dashed line
    axes[i].plot(test_dates[washout:], y_pred_denorm[:, i], label='Predicted', color='blue', linestyle='--', linewidth=2)
    
    # Set y-label for each subplot with increased font size
    axes[i].set_ylabel(f'{variable_names[i]}', fontsize=30)

# Add shared x-label with increased font size
fig.text(0.5, 0.01, 'Years', ha='center', fontsize=34)

# Add legend to the first subplot with increased font size
axes[0].legend(loc='upper left', fontsize=20)

# Adjust spacing between subplots
plt.tight_layout(rect=[0, 0.05, 1, 0.95])

# Set x-axis limits (adjust range as needed)
plt.xlim(test_dates.iloc[10], test_dates.iloc[350])

# Rotate x-tick labels for clarity
for ax in axes:
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

# Show the plot
# plt.savefig('sun_pred.eps',dpi=300)
plt.show()

**Prediction using conventional ESN**

In [ ]:
import time
import tracemalloc

# Initialize the Echo State Network with a single ring
n_inputs = 3  # Two inputs: x1 and x2
n_reservoir = 300
# Number of reservoir units
n_outputs = 3  # Two outputs: x1 and x2
spectral_radius = 0.9
leak_rate = 1.0
input_scaling = 1.0
input_connectivity = 0.2
rc_connectivity = 0.1
ridge_lambda = 1e-8  # Regularization
leak_rate = 1.0
seed = 12340
washout = 100

# Start tracing memory
tracemalloc.start()


start_train = time.time()


esn = EchoStateNetwork(
    n_inputs, n_reservoir, n_outputs, spectral_radius, leak_rate, 
    input_scaling, rc_connectivity, input_connectivity, seed
)
esn.fit(X_train, y_train, ridge_lambda=ridge_lambda, reset=True,washout=washout)

end_train = time.time()
train_time = end_train - start_train
print(f"Training Time: {train_time:.6f} seconds")



y_pred = esn.predict(X_test, reset=True,washout=washout)
nrmse = compute_nrmse(y_pred, y_test[washout:])
avg_nrmse = np.mean(nrmse)

# Get current and peak memory
current, peak = tracemalloc.get_traced_memory()
print(f"Current memory usage: {current / 1e6:.2f} MB")
print(f" Peak memory usage: {peak / 1e6:.2f} MB")

# Stop tracing
tracemalloc.stop()